# Employee Promotion Prediction — WNS Analytics Wizard 2018

**Portfolio project | Binary classification | Evaluation: F1-score**

## Business problem
Predict whether an employee will be promoted using historical performance, training, tenure and demographic attributes. The goal is to provide an earlier decision-support signal; model output should not replace a fair, human-led promotion process.

## Project workflow
1. Load and audit the supplied train, test and sample-submission files.
2. Explore target imbalance, missingness, numeric distributions and relationships.
3. Build reproducible preprocessing and feature engineering.
4. Compare Logistic Regression, Random Forest, XGBoost and CatBoost.
5. Tune a strong tree-based model with stratified cross-validation.
6. Select a decision threshold from out-of-fold (OOF) training predictions.
7. Evaluate with F1, precision, recall, PR-AUC and confusion matrix.
8. Explain model behavior with SHAP and export the competition submission.

> **Data caveat:** The business description mentions a KPI eligibility rule above 60%, but the provided field is `KPIs_met >80%`. These are not interchangeable. This notebook uses only the supplied field and does not fabricate a 60% KPI value.

## 0. Environment

Install dependencies in a terminal or notebook cell. XGBoost, CatBoost and SHAP are included for the requested model comparison and interpretation.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn scikit-learn xgboost catboost shap

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    RandomizedSearchCV, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, average_precision_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import joblib

RANDOM_STATE = 42
TRAIN_PATH = "train_LZdllcl.csv"
TEST_PATH = "test_2umaH9m.csv"
SAMPLE_PATH = "sample_submission_M0L0uXE.csv"

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

## 2. Load data and perform quality checks

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

TARGET = "is_promoted"
ID_COL = "employee_id"

print("Train:", train.shape, "| Test:", test.shape, "| Sample:", sample.shape)
display(train.head())
display(train.info())
display(train[TARGET].value_counts(dropna=False).rename("count").to_frame())
print("Target positive rate:", round(train[TARGET].mean(), 4))
print("Duplicate employee IDs in train:", train[ID_COL].duplicated().sum())
print("Duplicate employee IDs in test:", test[ID_COL].duplicated().sum())
print("Train columns missing from test:", sorted(set(train.columns) - set(test.columns) - {TARGET}))
print("Test columns missing from train:", sorted(set(test.columns) - set(train.columns)))

## 3. Exploratory Data Analysis — 10 useful visualizations

The charts are designed to answer business and modeling questions: class imbalance, missingness, age/tenure patterns, training intensity, ratings, KPI achievement, awards, department variation and training-score patterns. These are descriptive associations, not causal effects.

In [ ]:
# Plot 1 — Target class distribution
plt.figure(figsize=(6,4))
ax = sns.countplot(data=train, x=TARGET)
ax.set_title("1. Promotion target distribution")
ax.set_xlabel("Promoted (0 = No, 1 = Yes)")
ax.set_ylabel("Employees")
plt.tight_layout()
plt.show()

# Plot 2 — Missing values by column
missing_pct = train.isna().mean().mul(100).sort_values(ascending=False)
plt.figure(figsize=(9,4))
sns.barplot(x=missing_pct.values, y=missing_pct.index)
plt.title("2. Missing values by feature (%)")
plt.xlabel("Missing values (%)")
plt.tight_layout()
plt.show()

# Plot 3 — Age distribution by promotion
plt.figure(figsize=(8,4))
sns.histplot(data=train, x="age", hue=TARGET, bins=25, stat="density",
             common_norm=False, element="step")
plt.title("3. Age distribution by promotion outcome")
plt.tight_layout()
plt.show()

# Plot 4 — Length of service by promotion
plt.figure(figsize=(8,4))
sns.boxplot(data=train, x=TARGET, y="length_of_service")
plt.title("4. Length of service by promotion outcome")
plt.tight_layout()
plt.show()

# Plot 5 — Average training score by promotion
plt.figure(figsize=(8,4))
sns.kdeplot(data=train, x="avg_training_score", hue=TARGET, common_norm=False, fill=False)
plt.title("5. Average training score by promotion outcome")
plt.tight_layout()
plt.show()

# Plot 6 — Previous year rating by promotion
plt.figure(figsize=(8,4))
sns.countplot(data=train, x="previous_year_rating", hue=TARGET)
plt.title("6. Previous-year rating vs promotion")
plt.tight_layout()
plt.show()

# Plot 7 — KPI achievement by promotion
plt.figure(figsize=(7,4))
sns.countplot(data=train, x="KPIs_met >80%", hue=TARGET)
plt.title("7. KPI achievement (>80%) vs promotion")
plt.xlabel("KPIs met >80% (0/1)")
plt.tight_layout()
plt.show()

# Plot 8 — Awards by promotion
plt.figure(figsize=(7,4))
sns.countplot(data=train, x="awards_won?", hue=TARGET)
plt.title("8. Awards won vs promotion")
plt.xlabel("Awards won (0/1)")
plt.tight_layout()
plt.show()

# Plot 9 — Promotion rate by department
dept_rate = train.groupby("department")[TARGET].mean().sort_values(ascending=False)
plt.figure(figsize=(9,5))
sns.barplot(x=dept_rate.values, y=dept_rate.index)
plt.title("9. Promotion rate by department")
plt.xlabel("Observed promotion rate")
plt.tight_layout()
plt.show()

# Plot 10 — Number of trainings vs promotion
plt.figure(figsize=(8,4))
sns.countplot(data=train, x="no_of_trainings", hue=TARGET)
plt.title("10. Number of trainings vs promotion")
plt.tight_layout()
plt.show()

## 4. Feature engineering and train/test preparation

`employee_id` is excluded because it is an identifier, not a defensible employee-performance signal. Missing numerical values are median-imputed; missing categorical values are most-frequent-imputed. One-hot encoding is fitted inside each model pipeline to avoid preprocessing leakage.

Engineered variables are simple ratios/interactions. They are hypotheses to validate, not guaranteed improvements.

In [ ]:
def add_features(frame):
    frame = frame.copy()

    if {"no_of_trainings", "length_of_service"}.issubset(frame.columns):
        frame["trainings_per_service_year"] = (
            frame["no_of_trainings"] /
            frame["length_of_service"].replace(0, np.nan)
        )

    if {"age", "length_of_service"}.issubset(frame.columns):
        frame["estimated_career_start_age"] = (
            frame["age"] - frame["length_of_service"]
        )

    if {"previous_year_rating", "avg_training_score"}.issubset(frame.columns):
        frame["rating_x_training_score"] = (
            frame["previous_year_rating"] * frame["avg_training_score"]
        )

    return frame

X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

X = add_features(X)
X_test = add_features(X_test).reindex(columns=X.columns)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:  # compatibility with older scikit-learn
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot)
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 5. Baseline model comparison

Compare four requested algorithms using the same preprocessing and the same stratified folds. F1 is reported at the default 0.50 threshold; average precision (PR-AUC) is useful when the positive class is rare. These cross-validation scores are more informative than a single train/validation split.

In [ ]:
positive_rate = y.mean()
scale_pos_weight = (1 - positive_rate) / positive_rate

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", C=1.0
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.04,
        subsample=0.85, colsample_bytree=0.85,
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=700, depth=6, learning_rate=0.04,
        loss_function="Logloss", auto_class_weights="Balanced",
        verbose=False, random_seed=RANDOM_STATE, allow_writing_files=False
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"f1": "f1", "precision": "precision", "recall": "recall",
           "pr_auc": "average_precision"}

comparison_rows = []
pipelines = {}

for name, estimator in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    pipelines[name] = pipe
    scores = cross_validate(
        pipe, X, y, cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False
    )
    comparison_rows.append({
        "Model": name,
        "F1_mean": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "PR_AUC": scores["test_pr_auc"].mean()
    })

comparison = pd.DataFrame(comparison_rows).sort_values("F1_mean", ascending=False)
display(comparison)

## 6. Hyperparameter tuning

Tune CatBoost with randomized search and stratified CV. The search uses F1 at threshold 0.50 as the selection metric; later, the classification threshold is independently optimized using OOF probabilities. For a larger compute budget, increase `n_iter`.

In [ ]:
cat_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", CatBoostClassifier(
        loss_function="Logloss",
        auto_class_weights="Balanced",
        verbose=False,
        random_seed=RANDOM_STATE,
        allow_writing_files=False,
        thread_count=-1
    ))
])

param_distributions = {
    "model__iterations": [500, 800, 1100],
    "model__depth": [4, 5, 6, 7, 8],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
    "model__l2_leaf_reg": [3, 5, 7, 9]
}

search = RandomizedSearchCV(
    estimator=cat_pipe,
    param_distributions=param_distributions,
    n_iter=12,  # raise to 30-50 for a more thorough search
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
    refit=True
)
search.fit(X, y)

print("Best CV F1 @ 0.50:", search.best_score_)
print("Best parameters:", search.best_params_)
best_pipeline = search.best_estimator_

## 7. OOF predictions and F1 threshold optimization

Do not tune a threshold on the competition test set. Generate cross-validated probabilities for the training rows, then choose the threshold that maximizes OOF F1. This reduces the risk of selecting a threshold from in-sample predictions.

In [ ]:
oof_prob = cross_val_predict(
    best_pipeline, X, y, cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

threshold_grid = np.arange(0.05, 0.81, 0.005)
threshold_scores = [
    f1_score(y, (oof_prob >= t).astype(int)) for t in threshold_grid
]
best_threshold = float(threshold_grid[int(np.argmax(threshold_scores))])
oof_pred = (oof_prob >= best_threshold).astype(int)

print("Selected threshold:", round(best_threshold, 3))
print("OOF F1:", round(f1_score(y, oof_pred), 5))
print("OOF precision:", round(precision_score(y, oof_pred, zero_division=0), 5))
print("OOF recall:", round(recall_score(y, oof_pred, zero_division=0), 5))
print("OOF PR-AUC:", round(average_precision_score(y, oof_prob), 5))
print("OOF ROC-AUC:", round(roc_auc_score(y, oof_prob), 5))
print(classification_report(y, oof_pred, zero_division=0))

# Plot 1 — Threshold vs F1
plt.figure(figsize=(8,4))
plt.plot(threshold_grid, threshold_scores)
plt.axvline(best_threshold, linestyle="--", label=f"Best = {best_threshold:.3f}")
plt.xlabel("Classification threshold")
plt.ylabel("OOF F1-score")
plt.title("F1-score across classification thresholds")
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2 — Confusion matrix
ConfusionMatrixDisplay.from_predictions(y, oof_pred, values_format="d")
plt.title("OOF confusion matrix")
plt.tight_layout()
plt.show()

# Plot 3 — Precision-recall curve
precision_curve, recall_curve, _ = precision_recall_curve(y, oof_prob)
plt.figure(figsize=(6,5))
plt.plot(recall_curve, precision_curve)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Out-of-fold precision-recall curve")
plt.tight_layout()
plt.show()

## 8. SHAP explainability

SHAP explains how model features move predictions relative to the model's baseline. The following code explains a sample of rows to keep runtime manageable. Since preprocessing is inside the pipeline, SHAP is calculated on the transformed feature matrix and the CatBoost model.

Interpretation caution: SHAP describes model behavior, not causation or a fair/valid reason to promote or reject a person. Review sensitive attributes and potential proxies carefully before any real employment use.

In [ ]:
import shap

# Transform data using the fitted preprocessing pipeline
fitted_prep = best_pipeline.named_steps["prep"]
fitted_model = best_pipeline.named_steps["model"]

X_transformed = fitted_prep.transform(X)
feature_names = fitted_prep.get_feature_names_out()

# Use a reproducible subset for speed
rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(1000, X_transformed.shape[0])
sample_idx = rng.choice(X_transformed.shape[0], size=sample_size, replace=False)
X_shap = X_transformed[sample_idx]

explainer = shap.TreeExplainer(fitted_model)
shap_values = explainer.shap_values(X_shap)

# CatBoost binary classifier SHAP output is usually (rows, features).
# Handle list output from some SHAP versions.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    max_display=20,
    show=True
)

# Mean absolute SHAP values as a ranked table
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance.head(20))

## 9. Fit final model and create competition submission

The tuned pipeline is refitted on all labeled training rows. Predictions are made for the competition test data using the OOF-selected threshold. The sample submission is used to preserve the exact expected employee ID order.

In [ ]:
best_pipeline.fit(X, y)

test_prob = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

predictions = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: test_pred
})

submission = sample[[ID_COL]].merge(
    predictions, on=ID_COL, how="left", validate="one_to_one"
)

if submission[TARGET].isna().any():
    raise ValueError("Some sample submission employee IDs were not matched.")

submission[TARGET] = submission[TARGET].astype(int)
submission.to_csv("submission.csv", index=False)

joblib.dump(
    {"pipeline": best_pipeline, "threshold": best_threshold},
    "wns_promotion_model.joblib"
)

comparison.to_csv("model_comparison.csv", index=False)
shap_importance.to_csv("shap_feature_importance.csv", index=False)

print("Submission saved:", submission.shape)
print(submission.head())
print(submission[TARGET].value_counts())

## 10. Business interpretation and project conclusions

Use your actual results to complete this section:

- Best model and cross-validation F1: **[insert measured result]**
- OOF threshold and OOF F1: **[insert measured result]**
- Precision/recall trade-off: **[explain using measured values]**
- Most influential model features: **[insert top SHAP features]**
- Operational recommendation: use the model as an early-warning/decision-support tool, followed by structured human review and documented promotion criteria.

### Limitations and responsible use
- The dataset is historical and may encode past organizational bias.
- Demographic fields such as age and gender should not be used as promotion decision criteria. Consider excluding them and comparing performance, and conduct fairness analysis before any real-world deployment.
- SHAP importance is not causal evidence.
- The available KPI field represents >80%, not the stated >60% checkpoint eligibility.
- A high F1-score does not establish that the model is fair, calibrated or suitable for employment decisions.